# Hotel Booking Cancellation & Overbooking Decision
## 06 — Model Training with Training-Only Cross-Validation

Train the five required model families under one leakage-safe protocol. Model selection and Decision Tree depth selection use **training data only**. The held-out test set is deliberately not evaluated in this notebook; final test evaluation belongs in Notebook 07.

## 1. Imports

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    cross_validate,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.project_utils import (
    RANDOM_STATE,
    CV_FOLDS,
    TARGET,
    BOOKING_TIME_EXCLUSIONS,
    prepare_model_data,
    stratified_train_test_split,
    make_cv,
    make_random_forest,
    build_preprocessors,
)

cv = make_cv()

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

print("Repository root:", REPO_ROOT)
print("Random state:", RANDOM_STATE)
print("CV folds:", CV_FOLDS)


## 2. Load the Canonical Cleaned Dataset

Notebook 03 is the single source of truth for raw-data cleaning. This notebook therefore consumes the generated cleaned dataset rather than independently repeating cleaning rules.


In [ ]:
DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "hotel_bookings_cleaned.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Processed dataset not found. Run Notebook 03 first."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Cleaned shape:", df.shape)

assert df.shape == (119209, 30)


## 3. Validate the Cleaned Input

Before modelling, the notebook verifies that direct outcome leakage is absent, missing values have been resolved, and invalid negative ADR records are not present.


In [ ]:
direct_leakage = [
    "reservation_status",
    "reservation_status_date",
]

remaining = [
    c for c in direct_leakage
    if c in df.columns
]

print("Direct outcome leakage remaining:", remaining)
print("Remaining missing values:", int(df.isna().sum().sum()))
print("Maximum ADR retained:", df["adr"].max())

assert not remaining
assert int(df.isna().sum().sum()) == 0
assert (df["adr"] < 0).sum() == 0


## 4. Apply the Booking-Time Feature Policy

The shared `prepare_model_data()` function applies the stricter booking-time feature policy used consistently by Notebooks 05–08.

In addition to the direct outcome leakage removed in Notebook 03, the predictive feature set excludes:

- `assigned_room_type`
- `booking_changes`
- `days_in_waiting_list`

These fields may contain information that develops after the original reservation and are therefore not considered safe inputs for an at-booking cancellation decision.


In [ ]:
df_model = prepare_model_data(df)

forbidden = [
    "reservation_status",
    "reservation_status_date",
    *BOOKING_TIME_EXCLUSIONS,
]

remaining_forbidden = [
    c for c in forbidden
    if c in df_model.columns
]

print(
    "Booking-time exclusions:",
    BOOKING_TIME_EXCLUSIONS,
)
print(
    "Forbidden modelling columns remaining:",
    remaining_forbidden,
)
print("Model-ready shape:", df_model.shape)

assert not remaining_forbidden
assert df_model.shape == (119209, 26)


## 5. Canonical Feature Engineering

The shared modelling transformation creates the same booking-time-safe engineered features demonstrated in Notebook 05. Centralising this logic prevents training and evaluation notebooks from silently using different feature definitions.


In [ ]:
engineered_features = [
    "total_nights",
    "total_guests",
    "total_previous_bookings",
    "has_special_request",
]

print("Engineered features:")
for feature in engineered_features:
    print(feature, "->", feature in df_model.columns)

assert all(
    feature in df_model.columns
    for feature in engineered_features
)


## 6. High-Cardinality `country` Policy

Rare country handling is performed inside the preprocessing pipeline. Country levels below 1% frequency are learned from each training fold and grouped into `Other`.

Because the grouping rule is fitted inside the pipeline, validation-fold information is not used to decide which country levels are rare.


In [ ]:
print(
    "Raw cleaned country levels:",
    df["country"].nunique()
)

print(
    "Country policy: levels below 1% of each "
    "training fold are grouped into 'Other' "
    "inside the preprocessing pipeline."
)


## 7. Canonical Held-Out Stratified Split

The shared project split uses 80% training and 20% held-out test data, stratified by `is_canceled`, with `random_state=42`.

Notebook 06 uses only the training partition for model selection and cross-validation. The held-out test partition is reserved for Notebook 07.


In [ ]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

X_train, X_test, y_train, y_test = (
    stratified_train_test_split(X, y)
)

print("Train:", X_train.shape)
print("Held-out test:", X_test.shape)
print(
    "Train cancellation %:",
    round(y_train.mean() * 100, 2),
)
print(
    "Test cancellation %:",
    round(y_test.mean() * 100, 2),
)

print(
    "Held-out test is reserved for Notebook 07; "
    "no test metrics are computed here."
)

assert X_train.shape == (95367, 25)
assert X_test.shape == (23842, 25)


## 8. Shared Preprocessing Pipelines

The exact preprocessing implementation is imported from `src/project_utils.py`, matching Notebook 05 and the later evaluation/decision notebooks.


## 9. Cross-Validation Protocol

All five model families are compared using the same shuffled 5-fold stratified training cross-validation protocol.

ROC-AUC and Average Precision (PR-AUC) are the principal ranking metrics. Accuracy, precision, recall, and F1 are also reported for completeness.

The held-out test set is not used in this comparison.


In [ ]:
(
    preprocessor,
    boost_preprocessor,
    numerical_features,
    categorical_features,
) = build_preprocessors(X_train)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

assert len(numerical_features) == 16
assert len(categorical_features) == 9


## 10. Decision Tree Training-Only Tuning

Decision Tree depth and minimum leaf size are selected using 5-fold cross-validation on the training partition with ROC-AUC as the tuning objective.

The resulting parameters are then used in the common training-CV model comparison. Final unbiased model performance is reported separately on the untouched held-out test set in Notebook 07.


In [ ]:
tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ),
    ),
])

tree_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, None],
    "model__min_samples_leaf": [1, 5, 10],
}

tree_search = GridSearchCV(
    tree_pipeline,
    tree_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

tree_search.fit(X_train, y_train)

BEST_TREE_DEPTH = tree_search.best_params_[
    "model__max_depth"
]
BEST_TREE_MIN_LEAF = tree_search.best_params_[
    "model__min_samples_leaf"
]

print(
    "Best tree parameters:",
    tree_search.best_params_,
)
print(
    "Best training-CV ROC-AUC:",
    round(tree_search.best_score_, 4),
)


## 11. Define the Five Required Model Families

The comparison includes the five required classifier families:

1. Logistic Regression
2. K-Nearest Neighbours
3. Decision Tree
4. Random Forest
5. Histogram Gradient Boosting

KNN with `k=5` is treated as a standard untuned baseline. Random Forest uses the project's canonical 200-tree configuration so training, final evaluation, and threshold analysis evaluate the same model definition.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    # Standard untuned baseline.
    "KNN": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=5,
                n_jobs=-1,
            ),
        ),
    ]),

    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=BEST_TREE_DEPTH,
                min_samples_leaf=BEST_TREE_MIN_LEAF,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", make_random_forest()),
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", boost_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE
            ),
        ),
    ]),
}

print("Model families:")
for name in models:
    print("-", name)

rf = models["Random Forest"].named_steps["model"]

print(
    "Canonical Random Forest:",
    rf.n_estimators,
    "trees",
)

assert rf.n_estimators == 200


## 12. Compare All Five Models Using Training-Only Cross-Validation

No test-set predictions are made in this section.

In [ ]:
cv_rows = []

for name, model in models.items():
    print("Cross-validating:", name)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {"Model": name}

    for metric in scoring:
        values = scores["test_" + metric]

        row[metric.upper()] = values.mean()
        row[metric.upper() + "_STD"] = values.std()

    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows)

cv_results = (
    cv_results
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nFive-model training CV results:")
print(
    cv_results
    .round(4)
    .to_string(index=False)
)

assert len(cv_results) == 5
assert cv_results.isna().sum().sum() == 0


## 13. Visual Training-CV Comparison

In [ ]:
plot_cols = [
    "ACCURACY",
    "PRECISION",
    "RECALL",
    "F1",
    "ROC_AUC",
    "PR_AUC",
]

ax = (
    cv_results
    .set_index("Model")[plot_cols]
    .plot(
        kind="bar",
        figsize=(12, 6),
    )
)

plt.title(
    "Five-Model Comparison — "
    "5-Fold Training Cross-Validation"
)
plt.ylabel("Mean CV score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 14. Fit Final Candidate Models on the Full Training Set

After the CV comparison, each pipeline is fitted on all training rows so Notebook 07 can perform the **single final held-out test evaluation** under one identical protocol.

In [ ]:
fitted_models = {}

for name, model in models.items():
    print("Fitting final training model:", name)
    model.fit(X_train, y_train)
    fitted_models[name] = model

print(
    "All five candidate models fitted on "
    "training data only."
)

assert set(fitted_models) == set(models)


## 15. Training-Stage Conclusion

- Five required model families were compared under the same stratified 5-fold training-CV protocol.
- KNN receives scaled numerical features through its pipeline.
- Decision Tree depth/minimum leaf size were selected by training-only CV.
- Booking-time leakage variables were excluded before modelling.
- Rare `country` categories are explicitly grouped into `Other` inside the pipeline.
- **No held-out test metrics were used to choose a model in this notebook.**

### Next step
Notebook 07 should evaluate all five fitted model families once on the held-out test set, produce the required ROC/PR curves and confusion matrices, and compare Random Forest feature importance with Logistic Regression coefficients.